# Generacion de Embeddings Qwen2-VL-2B

**Qwen/Qwen2-VL-2B-Instruct** — LLM multimodal generativo (Alibaba, 2024)

## Diferencias clave vs CLIP/BLIP

- Qwen2-VL **NO es contrastivo**, es un LLM generativo multimodal.
- Extraemos embeddings haciendo **mean pooling** sobre los hidden states.
- Dim imagen: **1536** (vision encoder de Qwen)
- Dim texto: **1536** (LLM hidden size)
- L2-normalizados al final para uso con distancia coseno.

**Importante para tu tesis:** los embeddings de imagen y texto de Qwen2-VL NO viven en el mismo espacio compartido (a diferencia de CLIP/BLIP). Por eso comparar 'BBC News con CLIP' vs 'BBC News con Qwen2-VL' es valido (mismo dataset, mismo modelo), pero NO se pueden mezclar embeddings de Qwen entre modalidades.

## Tiempo estimado

T4 GPU (gratis): ~30-45 minutos totales (mucho mas lento que CLIP/BLIP).

## Pasos

1. **Runtime → Change runtime type → GPU (T4 o mejor)**
2. **Runtime → Run all**
3. Descargar `qwen_vl_embeddings.zip` al final
4. En tu PC, descomprimir en `clustering-python/embeddings/`

In [ ]:
# CELDA 1: Instalar dependencias
# Qwen2-VL necesita transformers >= 4.45 y qwen-vl-utils
!pip install -q --upgrade datasets huggingface_hub
!pip install -q transformers torchvision scikit-learn pillow qwen-vl-utils accelerate

In [ ]:
# CELDA 2: Imports y verificacion de GPU
import os, time, zipfile
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('ADVERTENCIA: Sin GPU este modelo NO funcionara razonablemente.')

In [ ]:
# CELDA 3: Cargar modelo Qwen2-VL-2B (en fp16 para ahorrar VRAM)
MODEL_NAME = 'Qwen/Qwen2-VL-2B-Instruct'
print('Cargando', MODEL_NAME, '...')
print('(Primera vez tarda ~3-5 min descargando ~5 GB)')

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
).eval()

processor = AutoProcessor.from_pretrained(MODEL_NAME)

# Dim de los hidden states
EMBED_DIM = model.config.hidden_size
print('Modelo listo. Dimension de embedding:', EMBED_DIM)
print(f'VRAM ocupada: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

In [ ]:
# CELDA 4: Funciones de embedding
# Estrategia: forward pass + mean pooling sobre los hidden states de la ultima capa

@torch.inference_mode()
def embed_texts(texts, batch_size=8, max_length=512):
    """
    Embeddings de texto con Qwen2-VL.
    
    Para textos puros (sin imagen) construimos un prompt del estilo Qwen,
    pasamos por el modelo y hacemos mean pooling sobre los tokens validos.
    
    Devuelve (N, hidden_size) float32 L2-normalizados.
    Batch size pequeño (8) porque el modelo es grande.
    """
    all_embs = []
    total = len(texts)
    
    for i in range(0, total, batch_size):
        batch = list(texts[i:i+batch_size])
        
        # Construir prompts en formato Qwen (sin imagen)
        messages_batch = [
            [{'role': 'user', 'content': [{'type': 'text', 'text': t}]}]
            for t in batch
        ]
        
        # Aplicar plantilla de chat
        texts_formatted = [
            processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
            for m in messages_batch
        ]
        
        # Tokenizar con truncado
        inputs = processor(
            text=texts_formatted,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=max_length,
        ).to(device)
        
        # Forward pass solicitando hidden states
        outputs = model(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )
        
        # Tomar hidden states de la ultima capa: (batch, seq_len, hidden_size)
        last_hidden = outputs.hidden_states[-1]
        
        # Mean pooling enmascarado (ignorar tokens de padding)
        mask = inputs.attention_mask.unsqueeze(-1).float()  # (batch, seq, 1)
        summed = (last_hidden * mask).sum(dim=1)            # (batch, hidden)
        counts = mask.sum(dim=1).clamp(min=1)               # (batch, 1)
        emb = summed / counts                                # mean pooling
        
        # L2 normalizar (para distancia coseno)
        emb = torch.nn.functional.normalize(emb, p=2, dim=-1)
        
        all_embs.append(emb.float().cpu().numpy())
        
        if (i // batch_size) % 5 == 0:
            print(f'  {min(i+batch_size, total)}/{total}', end='\r')
    
    return np.concatenate(all_embs, axis=0)


@torch.inference_mode()
def embed_images(images, batch_size=4):
    """
    Embeddings de imagen con Qwen2-VL.
    
    Pasamos la imagen con un prompt minimo y hacemos mean pooling sobre
    los hidden states de la ultima capa.
    
    Devuelve (N, hidden_size) float32 L2-normalizados.
    Batch size muy pequeño (4) porque imagen + LLM ocupa mucha VRAM.
    """
    all_embs = []
    total = len(images)
    
    for i in range(0, total, batch_size):
        batch = images[i:i+batch_size]
        batch = [img.convert('RGB') if img.mode != 'RGB' else img for img in batch]
        
        # Construir prompts: una imagen + texto minimo
        messages_batch = [
            [{'role': 'user', 'content': [
                {'type': 'image'},
                {'type': 'text', 'text': 'Describe.'},
            ]}]
            for _ in batch
        ]
        
        texts_formatted = [
            processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
            for m in messages_batch
        ]
        
        inputs = processor(
            text=texts_formatted,
            images=batch,
            return_tensors='pt',
            padding=True,
        ).to(device)
        
        outputs = model(
            **inputs,
            output_hidden_states=True,
            return_dict=True,
        )
        
        last_hidden = outputs.hidden_states[-1]
        
        # Mean pooling enmascarado
        mask = inputs.attention_mask.unsqueeze(-1).float()
        summed = (last_hidden * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        emb = summed / counts
        
        emb = torch.nn.functional.normalize(emb, p=2, dim=-1)
        
        all_embs.append(emb.float().cpu().numpy())
        
        if (i // batch_size) % 5 == 0:
            print(f'  {min(i+batch_size, total)}/{total} imagenes', end='\r')
        
        # Liberar memoria periodicamente
        if i % 100 == 0:
            torch.cuda.empty_cache()
    
    return np.concatenate(all_embs, axis=0)

In [ ]:
# CELDA 5: Helper para guardar .npz
OUT_DIR = Path('embeddings/qwen_vl')
OUT_DIR.mkdir(parents=True, exist_ok=True)

def save_npz(name, X, y, class_names, source):
    """Guarda embeddings como .npz con metadata."""
    out_path = OUT_DIR / (name + '.npz')
    metadata = {
        'source': source, 'model': MODEL_NAME,
        'dim': int(X.shape[1]), 'n': int(X.shape[0]),
        'n_classes': int(len(np.unique(y))),
    }
    np.savez_compressed(
        out_path,
        X=X.astype(np.float32),
        y=np.asarray(y, dtype=np.int64),
        class_names=np.asarray(class_names, dtype=object),
        metadata=np.asarray([metadata], dtype=object),
    )
    size_mb = out_path.stat().st_size / 1024**2
    norms = np.linalg.norm(X, axis=1)
    norm_ok = np.allclose(norms, 1.0, atol=1e-4)
    print(f'  [OK] {name:28s} N={metadata["n"]:5d} D={metadata["dim"]:4d} K={metadata["n_classes"]:3d} {size_mb:.2f}MB L2={"OK" if norm_ok else "FALLO"}')

## TEXTO - Datasets 1 a 5

In [ ]:
# [1/8] 20 Newsgroups (5 clases)
from sklearn.datasets import fetch_20newsgroups

print('\n[1/8] 20 Newsgroups (5 clases)')
t0 = time.time()
cats_5 = [
    'rec.sport.hockey',
    'rec.sport.baseball',
    'sci.med',
    'sci.space',
    'talk.politics.misc',
]
ds = fetch_20newsgroups(
    subset='all', categories=cats_5,
    remove=('headers', 'footers', 'quotes')
)
texts = ds.data
labels = list(ds.target)
class_names = list(ds.target_names)
print(f'  Distribucion: {dict(zip(class_names, [labels.count(i) for i in range(len(class_names))]))}')
X = embed_texts(texts)
save_npz('20ng_5classes', X, labels, class_names, 'sklearn:fetch_20newsgroups (5 cats)')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [2/8] BBC News
from datasets import load_dataset, concatenate_datasets

print('\n[2/8] BBC News')
t0 = time.time()
ds_train = load_dataset('SetFit/bbc-news', split='train')
try:
    ds_test = load_dataset('SetFit/bbc-news', split='test')
    ds = concatenate_datasets([ds_train, ds_test])
except Exception:
    ds = ds_train

texts = list(ds['text'])
labels = list(ds['label'])
if 'label_text' in ds.column_names:
    label_to_text = {}
    for lid, ltxt in zip(ds['label'], ds['label_text']):
        label_to_text[lid] = ltxt
    class_names = [label_to_text[i] for i in sorted(label_to_text.keys())]
else:
    class_names = [str(c) for c in sorted(set(labels))]
print(f'  Clases: {class_names}')
X = embed_texts(texts)
save_npz('bbc_news', X, labels, class_names, 'huggingface:SetFit/bbc-news')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [3/8] 20 Newsgroups (2 clases)
print('\n[3/8] 20 Newsgroups (2 clases)')
t0 = time.time()
cats_2 = ['alt.atheism', 'soc.religion.christian']
ds = fetch_20newsgroups(
    subset='all', categories=cats_2,
    remove=('headers', 'footers', 'quotes')
)
texts, labels = ds.data, list(ds.target)
class_names = list(ds.target_names)
X = embed_texts(texts)
save_npz('20ng_2classes', X, labels, class_names, 'sklearn:fetch_20newsgroups')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [4/8] 20 Newsgroups (3 clases)
print('\n[4/8] 20 Newsgroups (3 clases)')
t0 = time.time()
cats_3 = ['comp.graphics', 'rec.sport.hockey', 'sci.med']
ds = fetch_20newsgroups(
    subset='all', categories=cats_3,
    remove=('headers', 'footers', 'quotes')
)
texts, labels = ds.data, list(ds.target)
class_names = list(ds.target_names)
X = embed_texts(texts)
save_npz('20ng_3classes', X, labels, class_names, 'sklearn:fetch_20newsgroups')
print(f'  Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# [5/8] AG News (subsample 1000/clase = 4000 total)
print('\n[5/8] AG News (4k subsample)')
t0 = time.time()
ds_ag = load_dataset('ag_news', split='train')
ds_ag = ds_ag.shuffle(seed=42)

PER_CLASS = 1000
selected_idx = []
counts = {0: 0, 1: 0, 2: 0, 3: 0}
for i, label in enumerate(ds_ag['label']):
    if counts[label] < PER_CLASS:
        selected_idx.append(i)
        counts[label] += 1
    if all(c >= PER_CLASS for c in counts.values()):
        break

ds_sub = ds_ag.select(selected_idx)
texts = list(ds_sub['text'])
labels = list(ds_sub['label'])
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']
print(f'  Distribucion: {dict(zip(class_names, [labels.count(i) for i in range(4)]))}')
X = embed_texts(texts)
save_npz('ag_news_4k', X, labels, class_names, 'huggingface:ag_news (1k/class)')
print(f'  Tiempo: {time.time()-t0:.1f}s')

## IMAGENES - Datasets 6 a 8

⚠️ Las imagenes con Qwen2-VL son mas lentas que el texto porque cada imagen se procesa con el vision encoder + LLM. Estima ~5-10 min por dataset de 5000 imagenes.

In [ ]:
# [6/8] ORL Faces (Olivetti) - 400 imagenes
from sklearn.datasets import fetch_olivetti_faces

print('\n[6/8] ORL Faces')
t0 = time.time()
ds_oli = fetch_olivetti_faces(shuffle=False)

images = []
for img_arr in ds_oli.images:
    img = (img_arr * 255).astype(np.uint8)
    pil = Image.fromarray(img, mode='L').convert('RGB')
    images.append(pil)

labels = list(ds_oli.target)
class_names = [f'person_{i}' for i in range(40)]
X = embed_images(images)
save_npz('orl_faces', X, labels, class_names, 'sklearn:fetch_olivetti_faces')
print(f'  Tiempo: {time.time()-t0:.1f}s')

torch.cuda.empty_cache()

In [ ]:
# [7/8] MNIST (subsample 500/clase = 5000 total)
import torchvision

print('\n[7/8] MNIST (5k subsample)')
t0 = time.time()
mnist = torchvision.datasets.MNIST(
    root='/tmp/mnist', train=True, download=True
)

PER_CLASS = 500
targets = np.asarray(mnist.targets)
selected_idx = []
for c in range(10):
    idx_c = np.where(targets == c)[0][:PER_CLASS]
    selected_idx.extend(idx_c.tolist())

images, labels = [], []
for i in selected_idx:
    img, lab = mnist[i]
    images.append(img.convert('RGB'))
    labels.append(int(lab))

class_names = [str(i) for i in range(10)]
X = embed_images(images)
save_npz('mnist_5k', X, labels, class_names, 'torchvision:MNIST (500/class)')
print(f'  Tiempo: {time.time()-t0:.1f}s')

torch.cuda.empty_cache()

In [ ]:
# [8/8] Fashion-MNIST (subsample 500/clase = 5000 total)
print('\n[8/8] Fashion-MNIST (5k subsample)')
t0 = time.time()
fmnist = torchvision.datasets.FashionMNIST(
    root='/tmp/fmnist', train=True, download=True
)

PER_CLASS = 500
targets = np.asarray(fmnist.targets)
selected_idx = []
for c in range(10):
    idx_c = np.where(targets == c)[0][:PER_CLASS]
    selected_idx.extend(idx_c.tolist())

images, labels = [], []
for i in selected_idx:
    img, lab = fmnist[i]
    images.append(img.convert('RGB'))
    labels.append(int(lab))

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
              'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle_boot']
X = embed_images(images)
save_npz('fashion_mnist_5k', X, labels, class_names, 'torchvision:FashionMNIST (500/class)')
print(f'  Tiempo: {time.time()-t0:.1f}s')

torch.cuda.empty_cache()

## Resumen y descarga

In [ ]:
# Resumen de todo lo generado
print('\n=== RESUMEN FINAL ===')
print(f'{"Dataset":<28} {"N":>6} {"D":>5} {"K":>4} {"MB":>7} {"L2":>4}')
print('-' * 60)
total_mb = 0
all_ok = True
for f in sorted(OUT_DIR.glob('*.npz')):
    size_mb = f.stat().st_size / 1024**2
    total_mb += size_mb
    npz = np.load(f, allow_pickle=True)
    X, y = npz['X'], npz['y']
    norms = np.linalg.norm(X, axis=1)
    norm_ok = np.allclose(norms, 1.0, atol=1e-4)
    if not norm_ok:
        all_ok = False
    print(f'{f.stem:<28} {X.shape[0]:>6} {X.shape[1]:>5} {len(np.unique(y)):>4} {size_mb:>7.2f} {"OK" if norm_ok else "FALLO":>4}')

print('-' * 60)
print(f'{"TOTAL":28} {"":>6} {"":>5} {"":>4} {total_mb:>7.2f}')
print()
if all_ok:
    print('✓ Todos los archivos pasaron verificacion L2.')
else:
    print('⚠ Algunos archivos tienen problemas de normalizacion.')

In [ ]:
# Crear ZIP para descargar
ZIP_PATH = 'qwen_vl_embeddings.zip'
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in OUT_DIR.glob('*.npz'):
        zf.write(f, arcname=f'embeddings/qwen_vl/{f.name}')

zip_mb = os.path.getsize(ZIP_PATH) / 1024**2
print(f'ZIP creado: {ZIP_PATH} ({zip_mb:.2f} MB)')

In [ ]:
# Descargar al navegador
from google.colab import files
files.download(ZIP_PATH)

In [ ]:
# (OPCIONAL) Guardar en Google Drive si el ZIP es grande o se cierra Colab
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# shutil.copy(ZIP_PATH, '/content/drive/MyDrive/qwen_vl_embeddings.zip')
# print('Copiado a Drive')

## Pasos siguientes en tu PC

1. Descomprimir `qwen_vl_embeddings.zip` en la raiz del proyecto `clustering-python/`
   - Quedara la estructura: `embeddings/qwen_vl/<dataset>.npz`

2. Verificar:
   ```cmd
   venv\Scripts\activate
   python embeddings_loader.py qwen_vl
   ```

3. Correr los 8 algoritmos con Qwen2-VL:
   ```cmd
   python testing.py 9 --model qwen_vl
   ```

4. Generar tabla comparativa con todos los modelos:
   ```cmd
   python build_results_table.py --all
   ```

## Notas para tu tesis

- **Qwen2-VL es generativo**, no contrastivo. Los embeddings son extraidos por mean pooling sobre los hidden states de la ultima capa del LLM.
- Las dimensiones (1536) son **mayores** que CLIP (512) y BLIP (256). Esto afecta los tiempos de los algoritmos basados en matrices N×N o ILP.
- Si los resultados de clustering son inferiores a CLIP/BLIP, esto es un **hallazgo cientifico legitimo** que puedes discutir: los modelos generativos no necesariamente producen los mejores embeddings para tareas no supervisadas.